# Mask2Former + PhenoBench: segmentacion de cultivo y maleza

Inferencia con pesos preentrenados de la Universidad de Bonn. **No se entrena nada.**

- **Arquitectura:** Cheng et al., *Masked-attention Mask Transformer for Universal Image Segmentation*, CVPR 2022 (arXiv 2112.01527)
- **Pesos y datos:** Weyler et al., *PhenoBench*, arXiv 2306.04557

---

### Antes de empezar

`Entorno de ejecucion > Cambiar tipo de entorno > GPU T4`

### Orden de ejecucion

Secciones 1 a 3 una sola vez. Al terminar la 3 hay que **reiniciar la sesion**.
Despues se ejecuta la seccion 0 y se sigue desde la 4.
Los archivos de `/content` sobreviven al reinicio; las variables no.

## 0. Reanudar despues de reiniciar

Ejecuta SOLO esta celda cuando vuelvas tras un reinicio de sesion, y continua en la seccion 4.
La primera vez, saltatela y ve directo a la seccion 1.

In [ ]:
M2F = '/content/phenobench-baselines/leaf_instance_segmentation/mask2former/src/Mask2Former'

import torch                          # primero torch: carga libc10.so en memoria
import MultiScaleDeformableAttention  # luego el kernel compilado
print('kernel deformable: ok')

import os
print('checkpoint presente:', os.path.exists('/content/phenobench_panoptic.pth'))

## 1. Dependencias

Se usa el torch que trae Colab. Fijar una version vieja no funciona: Colab corre
Python 3.13 y no existen ruedas de torch 2.1 para esa version.

In [ ]:
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'
!pip install -q opencv-python timm scipy ninja

import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())
print('gpu:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'ninguna')

## 2. Codigo fuente

In [ ]:
%cd /content
!git clone -q --recurse-submodules https://github.com/PRBonn/phenobench-baselines

M2F = '/content/phenobench-baselines/leaf_instance_segmentation/mask2former/src/Mask2Former'
print(M2F)

## 3. Parchear y compilar el kernel de atencion deformable

El pixel decoder de Mask2Former usa atencion deformable multiescala, implementada
como un kernel CUDA propio. El codigo es de 2021 y usa APIs de PyTorch que ya
no compilan, asi que hay que corregir dos cosas distintas en el mismo archivo:

| Linea | Codigo original | Problema | Correccion |
|---|---|---|---|
| 39, 105 | `value.type().is_cuda()` | Verifica si el tensor esta en GPU | `value.is_cuda()` |
| 69, 139 | `value.type()` dentro de `AT_DISPATCH` | Necesita un `ScalarType` | `value.scalar_type()` |

El orden importa: primero las verificaciones de GPU, luego el resto. Si se
reemplaza todo de una vez se rompen las lineas 39 y 105.

In [ ]:
import pathlib

cu = pathlib.Path(M2F) / 'mask2former/modeling/pixel_decoder/ops/src/cuda/ms_deform_attn_cuda.cu'
src = cu.read_text()

# 1) Verificaciones de tensor en GPU (lineas 39 y 105)
n1 = src.count('value.type().is_cuda()')
src = src.replace('value.type().is_cuda()', 'value.is_cuda()')

# 2) Despacho por tipo escalar (lineas 69 y 139)
n2 = src.count('value.type()')
src = src.replace('value.type()', 'value.scalar_type()')

cu.write_text(src)
print(f'is_cuda corregidos: {n1} (esperado 2)')
print(f'scalar_type corregidos: {n2} (esperado 2)')

In [ ]:
%cd {M2F}/mask2former/modeling/pixel_decoder/ops
!rm -rf build
!python setup.py build install 2>&1 | tail -15
%cd /content

In [ ]:
import MultiScaleDeformableAttention
print('compilado ok')

### Pesos (176 MB, servidor de la Universidad de Bonn)

In [ ]:
!wget -q --show-progress -O /content/phenobench_panoptic.pth \
  https://www.ipb.uni-bonn.de/html/projects/phenobench/panoptic_segmentation/Mask2former/model.pth
!ls -lh /content/phenobench_panoptic.pth

---

## >>> REINICIA LA SESION AQUI <<<

`Entorno de ejecucion > Reiniciar sesion`, luego ejecuta la **seccion 0** y sigue abajo.

## 4. Construir el modelo

Valores tomados del config verificado (`maskformer_plants.yaml` y su base):
ResNet-50, 3 clases, 100 queries, 10 capas de decoder, 256 dimensiones, 8 cabezas, 1024 px.

`PIXEL_MEAN` y `PIXEL_STD` estan comentados en el YAML, asi que rigen los valores
por defecto de Detectron2: media BGR y desviacion unitaria. Normalizar al estilo
ImageNet aqui arruina las predicciones aunque el modelo cargue sin errores.

In [ ]:
import sys, torch
sys.path.insert(0, M2F)

from detectron2.config import get_cfg
from detectron2.projects.deeplab import add_deeplab_config
from detectron2.engine import DefaultPredictor
from mask2former import add_maskformer2_config

CONFIG = f'{M2F}/configs/phenorobplants/panoptic-segmentation/maskformer2_R50_bs16_90k.yaml'
# Alternativa si el _BASE_ no se resuelve:
# CONFIG = '/content/phenobench-baselines/leaf_instance_segmentation/mask2former/configs/maskformer_plants.yaml'

cfg = get_cfg()
add_deeplab_config(cfg)
add_maskformer2_config(cfg)
cfg.merge_from_file(CONFIG)

cfg.MODEL.WEIGHTS = '/content/phenobench_panoptic.pth'
cfg.MODEL.DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
cfg.INPUT.MIN_SIZE_TEST = 1024
cfg.INPUT.MAX_SIZE_TEST = 1024
cfg.MODEL.MASK_FORMER.TEST.SEMANTIC_ON = True
cfg.MODEL.MASK_FORMER.TEST.INSTANCE_ON = True
cfg.MODEL.MASK_FORMER.TEST.PANOPTIC_ON = True
cfg.freeze()

predictor = DefaultPredictor(cfg)
print('\nModelo listo en', cfg.MODEL.DEVICE)

## 5. Datos de arquitectura para el informe

Estos son los numeros que hay que saber de memoria en la sustentacion.

In [ ]:
mf = cfg.MODEL.MASK_FORMER
m = predictor.model

print('Queries de objeto      :', mf.NUM_OBJECT_QUERIES)
print('Capas del decoder      :', mf.DEC_LAYERS, '(9 utiles + 1 para la perdida)')
print('Dimension oculta       :', mf.HIDDEN_DIM)
print('Cabezas                :', mf.NHEADS, '->', mf.HIDDEN_DIM // mf.NHEADS, 'dim por cabeza')
print('Feed-forward           :', mf.DIM_FEEDFORWARD)
print('Capas del pixel decoder:', cfg.MODEL.SEM_SEG_HEAD.TRANSFORMER_ENC_LAYERS, '(atencion deformable)')
print('Clases                 :', cfg.MODEL.SEM_SEG_HEAD.NUM_CLASSES, '(+1 "sin objeto")')
print('Backbone               :', cfg.MODEL.BACKBONE.NAME, '| ResNet-%d' % cfg.MODEL.RESNETS.DEPTH)
print('Parametros             : %.1f M' % (sum(p.numel() for p in m.parameters()) / 1e6))

pred = m.sem_seg_head.predictor
print('\nTensores del decoder:')
print('  query_feat  (las Q aprendidas) :', tuple(pred.query_feat.weight.shape))
print('  query_embed (pos. de las Q)    :', tuple(pred.query_embed.weight.shape))
print('  class_embed (cabeza de clase)  :', tuple(pred.class_embed.weight.shape))

## 6. Inferencia

**Empieza con una imagen del propio PhenoBench** (phenobench.ipb.uni-bonn.de), no con
una tuya. Solo asi puedes verificar contra la mascara de referencia que el mapeo de
clases de la seccion 7 es correcto. Despues prueba con imagenes de dron.

Sube el archivo con el panel de la izquierda y ajusta `RUTA`.

In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt, time

RUTA = '/content/mi_imagen.png'   # <-- cambia esto

img_bgr = cv2.imread(RUTA)        # Detectron2 espera BGR
assert img_bgr is not None, f'No se pudo leer {RUTA}'
print('Imagen:', img_bgr.shape)

t0 = time.time()
salida = predictor(img_bgr)
print('Inferencia: %.1f s' % (time.time() - t0))
print('Salidas disponibles:', list(salida.keys()))

## 7. Segmentacion semantica

El orden de clases es una suposicion razonable, no un dato confirmado.
Verificalo con una imagen anotada de PhenoBench antes de sacar conclusiones,
y si sale invertido, ajusta los diccionarios.

In [ ]:
NOMBRES = {0: 'suelo', 1: 'cultivo', 2: 'maleza'}
COLORES = {0: (0, 0, 0), 1: (0, 200, 0), 2: (220, 0, 0)}

img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
sem = salida['sem_seg'].argmax(0).cpu().numpy()

capa = np.zeros((*sem.shape, 3), dtype=np.uint8)
for k, c in COLORES.items():
    capa[sem == k] = c

fig, ax = plt.subplots(1, 3, figsize=(20, 7))
ax[0].imshow(img_rgb);  ax[0].set_title('Original')
ax[1].imshow(capa);     ax[1].set_title('Semantica')
ax[2].imshow(img_rgb);  ax[2].imshow(capa, alpha=0.5); ax[2].set_title('Superpuesto')
for a in ax: a.axis('off')
plt.tight_layout(); plt.savefig('/content/semantica.png', dpi=120); plt.show()

print('Cobertura por clase:')
for k, n in NOMBRES.items():
    print(f'  {n:<10} {(sem == k).sum() / sem.size * 100:5.1f} %')

## 8. Segmentacion de instancias

Cada planta por separado. Esto es lo que distingue a Mask2Former de una red de
segmentacion semantica: no solo dice que pixeles son maleza, sino **cuantas**
plantas de maleza hay y donde empieza y termina cada una.

In [ ]:
from collections import Counter

UMBRAL = 0.8   # mismo valor que OBJECT_MASK_THRESHOLD en el config

inst_todas = salida['instances'].to('cpu')
inst = inst_todas[inst_todas.scores > UMBRAL]
print(f'Instancias totales: {len(inst_todas)} | con score > {UMBRAL}: {len(inst)}')

if len(inst):
    clases = [NOMBRES.get(int(c), str(int(c))) for c in inst.pred_classes]
    print('Por clase:', dict(Counter(clases)))
    print('Score medio: %.3f' % float(inst.scores.mean()))

    capa_i = np.zeros((*img_rgb.shape[:2], 3), dtype=np.uint8)
    rng = np.random.default_rng(0)
    for msk in inst.pred_masks.numpy():
        capa_i[msk > 0.5] = rng.integers(60, 255, 3)   # mascaras vienen como float

    fig, ax = plt.subplots(1, 2, figsize=(14, 7))
    ax[0].imshow(img_rgb); ax[0].set_title('Original')
    ax[1].imshow(img_rgb); ax[1].imshow(capa_i, alpha=0.6)
    ax[1].set_title(f'{len(inst)} instancias')
    for a in ax: a.axis('off')
    plt.tight_layout(); plt.savefig('/content/instancias.png', dpi=120); plt.show()
else:
    print('Sin detecciones sobre el umbral. Con imagenes de otro cultivo o altura')
    print('de vuelo es esperable, y es un resultado valido para el analisis.')

## 9. Las 100 queries en crudo

El modelo **siempre** emite 100 predicciones, haya 3 plantas o 50. Las sobrantes
votan por la clase "sin objeto". Esta celda lo muestra, y es la respuesta directa
a la pregunta de como se genera la salida.

In [ ]:
# El decoder SIEMPRE emite 100 predicciones; las sobrantes votan 'sin objeto'.
print('Queries totales que emite el decoder :', cfg.MODEL.MASK_FORMER.NUM_OBJECT_QUERIES)
print('Queries que superaron el umbral      :', len(inst))
print('Queries descartadas                  :', cfg.MODEL.MASK_FORMER.NUM_OBJECT_QUERIES - len(inst))
print()
print('Por eso Mask2Former no necesita NMS: predice un conjunto de tamano fijo y')
print('las queries sobrantes se declaran vacias. El emparejamiento bipartito del')
print('entrenamiento es lo que les ensena a repartirse los objetos.')

## 10. Guardar resultados

Descarga las figuras para el informe.

In [ ]:
from google.colab import files
import os

for f in ['/content/semantica.png', '/content/instancias.png']:
    if os.path.exists(f):
        files.download(f)

## 11. Exportar resultados para la aplicacion local

Los pesos de PhenoBench no pueden ejecutarse en la app de Streamlit del equipo local,
porque dependen de Detectron2 y del kernel CUDA compilado en la seccion 3. Esta seccion
corre el modelo sobre un lote de imagenes y empaqueta las salidas para que la app las muestre.

1. Ejecuta antes las secciones 0 y 4 (modelo cargado en `predictor`).
2. Sube las imagenes a `/content/imagenes/`: una o dos del conjunto oficial de PhenoBench
   (para mostrar el modelo en su dominio) y las tuyas de dron.
3. Ejecuta las dos celdas siguientes. Se descarga `phenobench_export.zip`.
4. En tu equipo, descomprimelo dentro de `resultados/phenobench/` del repositorio.


In [ ]:
import os, json, time, glob
import cv2, numpy as np, matplotlib.pyplot as plt
from collections import Counter

ENTRADA = '/content/imagenes'
SALIDA  = '/content/phenobench_export'
os.makedirs(ENTRADA, exist_ok=True)
os.makedirs(SALIDA, exist_ok=True)

NOMBRES = {0: 'suelo', 1: 'cultivo', 2: 'maleza'}
COLORES = {0: (0, 0, 0), 1: (0, 200, 0), 2: (220, 0, 0)}
UMBRAL  = 0.8     # mismo valor que OBJECT_MASK_THRESHOLD del config

rutas = sorted(p for p in glob.glob(f'{ENTRADA}/*')
               if p.lower().endswith(('.jpg', '.jpeg', '.png', '.tif', '.tiff')))
assert rutas, f'No hay imagenes en {ENTRADA}. Subelas con el panel de archivos.'

resumen = {}
for ruta in rutas:
    img_bgr = cv2.imread(ruta)                 # Detectron2 espera BGR
    if img_bgr is None:
        print('[omitida]', ruta); continue

    t0 = time.time()
    salida = predictor(img_bgr)
    dt = time.time() - t0

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # Semantica: una clase por pixel
    sem = salida['sem_seg'].argmax(0).cpu().numpy()
    capa = np.zeros((*sem.shape, 3), dtype=np.uint8)
    for k, c in COLORES.items():
        capa[sem == k] = c

    # Instancias: solo las que superan el umbral de confianza
    inst_t = salida['instances'].to('cpu')
    inst = inst_t[inst_t.scores > UMBRAL]
    capa_i = np.zeros_like(img_rgb)
    rng = np.random.default_rng(0)
    for m in inst.pred_masks.numpy():
        capa_i[m > 0.5] = rng.integers(60, 255, 3)

    nombre = os.path.splitext(os.path.basename(ruta))[0]
    fig, ax = plt.subplots(1, 3, figsize=(21, 7))
    ax[0].imshow(img_rgb);                          ax[0].set_title('Original')
    ax[1].imshow(img_rgb); ax[1].imshow(capa, alpha=0.5)
    ax[1].set_title('Semantica: verde cultivo, rojo maleza')
    ax[2].imshow(img_rgb); ax[2].imshow(capa_i, alpha=0.6)
    ax[2].set_title(f'{len(inst)} instancias (score > {UMBRAL})')
    for a in ax: a.axis('off')
    plt.tight_layout()
    plt.savefig(f'{SALIDA}/{nombre}_phenobench.png', dpi=110, bbox_inches='tight')
    plt.close()

    # Estas claves son las que lee app.py: no cambiarlas
    resumen[nombre] = {
        'archivo_figura': f'{nombre}_phenobench.png',
        'tamano': [int(img_bgr.shape[1]), int(img_bgr.shape[0])],
        'tiempo_s': round(dt, 2),
        'cobertura_pct': {n: round(float((sem == k).mean() * 100), 2)
                          for k, n in NOMBRES.items()},
        'instancias_totales': int(len(inst_t)),
        'instancias_sobre_umbral': int(len(inst)),
        'por_clase': dict(Counter(NOMBRES.get(int(c), str(int(c)))
                                  for c in inst.pred_classes)),
        'score_medio': round(float(inst.scores.mean()), 3) if len(inst) else None,
    }
    print(f"{nombre:<30} {dt:5.2f} s  instancias: {len(inst):>3}  "
          f"cultivo {resumen[nombre]['cobertura_pct']['cultivo']:5.1f} %  "
          f"maleza {resumen[nombre]['cobertura_pct']['maleza']:5.1f} %")

with open(f'{SALIDA}/resumen.json', 'w', encoding='utf-8') as f:
    json.dump(resumen, f, ensure_ascii=False, indent=2)
print(f'\n{len(resumen)} imagenes exportadas a {SALIDA}')

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('/content/phenobench_export', 'zip', '/content/phenobench_export')
files.download('/content/phenobench_export.zip')